In [ ]:
%pip install "ray[default]"
%pip install python-dotenv

In [ ]:
import os 
import ray 

In [ ]:
import sys; sys.path.append("..")
import warnings; warnings.filterwarnings("ignore")
from dotenv import load_dotenv; load_dotenv(override=True)
%load_ext autoreload
%autoreload 2

In [ ]:
if ray.is_initialized():
    ray.shutdown()
ray.init(
    num_cpus= 4,
    object_store_memory=2 * 1024 * 1024 * 1024,
    runtime_env={"env_vars": {"USE_LIBUV": "0"}}
)

In [ ]:
ray.cluster_resources()

In [ ]:
num_workers = 2
resources_per_worker = {"CPU": 1, "GPU": 1}

In [ ]:
import os
from pathlib import Path

if os.path.exists("/efs"):
    EFS_DIR = f"/efs/shared_storage/MY-FIRST-MLOPS-PROJECT/{os.environ.get('Eva', 'default_user')}"
else:    
    EFS_DIR = str(Path(os.getcwd()).parent / "local_storage")
    os.makedirs(EFS_DIR, exist_ok=True)

print(f"Current active storage directory: {EFS_DIR}")

In [ ]:
import pandas as pd 

In [ ]:
dataset = "https://raw.githubusercontent.com/evasim/my-first-MLOPS-project/refs/heads/main/data/raw_dataset.csv"
df = pd.read_csv(dataset)
df.head()

In [ ]:
from sklearn.model_selection import train_test_split

In [ ]:
df.label.value_counts()

In [ ]:
test_size = 0.2 
train_df, val_df = train_test_split(df, stratify=df.label, test_size=test_size, random_state=42)

In [ ]:
train_df.label.value_counts()

In [ ]:
val_df.label.value_counts() * int((1 - test_size) / test_size)

In [ ]:
from collections import Counter 
import matplotlib.pyplot as plt 
import seaborn as sns; sns.set_theme()
import warnings; warnings.filterwarnings("ignore")
from wordcloud import WordCloud, STOPWORDS

In [ ]:
tags = Counter(df.label)
tags.most_common()

In [ ]:
import json 
import nltk 
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer
import re

In [ ]:
nltk.download("stopwords")
words = stopwords.words("english")

In [ ]:
df.head()

In [ ]:
def clean_text(text, stopwords=words):
    # change every words into lower case
    text = text.lower()

    text = re.sub(r"http\S+", "", text) # remove links (must run before punctuation gets spaced apart below)

    # removing stopwords such as "is", "the" and so on
    pattern = re.compile(r'\b(' + r"|".join(words)+ r")\b\s*")
    text = pattern.sub('', text)

    text = re.sub(r"([!\"'#$%&()*\+,-./:;<=>?@\\\[\]^_`{|}~])", r" \1 ", text) # add space 
    text = re.sub("[^A-Za-z0-9]+", " ", text) # remove other than words and numbers 
    text = re.sub(" +", " ", text) # remove all extra spaces 
    text = text.strip() # remove spaces at the start and at the end 

    return text

In [ ]:
ori_df = df.copy()
df.text = df.text.apply(clean_text)
print(f"{ori_df.text.values[0]}\n{df.text.values[0]}")

In [ ]:
df = df.dropna(subset=["label"]) 
df.head()

In [ ]:
label_decoder = {
    0: "World",
    1: "Sports",
    2: "Business",
    3: "Sci/Tech"
}

In [ ]:
import numpy as np 
from transformers import BertTokenizer

In [ ]:
_tokenizer_cache = {}

def get_tokenizer(model_name="allenai/scibert_scivocab_uncased"):
    # cache per-process so map_batches workers don't reload the tokenizer on every batch
    if model_name not in _tokenizer_cache:
        _tokenizer_cache[model_name] = BertTokenizer.from_pretrained(model_name, return_dict=False)
    return _tokenizer_cache[model_name]

def tokenize(batch):
    tokenizer = get_tokenizer()
    encoded = tokenizer(batch["text"].tolist(), return_tensors="np", padding = "longest")
    return dict(ids=encoded["input_ids"], masks=encoded["attention_mask"], targets=np.array(batch["label"]))

In [ ]:
tokenize(df.head(1))

In [ ]:
# combining all preprocessing steps into function
def preprocess(df):
    df["text"] = df.text.apply(clean_text)
    targets = tokenize(df)
    return targets 

In [ ]:
preprocess(df=train_df)

In [ ]:
from src.data import data, stratify_split
ray.data.DataContext.get_current().execution_options.preserve_order = True

In [ ]:
ds = ray.data.read_csv(dataset)
ds = ds.random_shuffle(seed=1234)
ds.take(1)


In [ ]:
test_size = 0.2
train_ds, val_ds = stratify_split(ds, stratify="label", test_size=test_size)

In [ ]:
label = train_ds.unique(column="label")
class_to_index = {label: i for i, label in enumerate(label)}

In [ ]:
sample_ds = train_ds.map_batches(preprocess, batch_format="pandas")
sample_ds.show(1)

In [ ]:
import os 
import random
import torch 
from ray.data.preprocessor import Preprocessor

def setting_seeds(seed=42):
    np.random.seed(seed)
    random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    eval("setattr(torch.backends.cudnn, 'deterministic', True)") # forces GPU to always perform calculation in exact sequence
    eval("setattr(torch.backends.cudnn, 'benchmark', False)") 
    os.environ["PYTHONHASHSEED"] = str(seed)

In [ ]:
def loading_data(num_samples=None):
    ds = ray.data.read_csv(dataset)
    ds = ds.random_shuffle(seed=1234)
    ds = ray.data.from_items(ds.take(num_samples)) if num_samples else ds
    return ds

In [ ]:
class CustomPreprocessor():
    """Custom Preprocess."""
    def __init__(self, label_decoder = None):
        self.label_decoder = label_decoder or {
            0: "World",
            1: "Sports",
            2: "Business",
            3: "Sci/Tech"
        }
        self.class_to_index = {v:k for k, v in self.label_decoder.items()}

    def fitting(self, ds):
        _ = ds.unique(column="label")
        return self
    
    def transforming(self, ds):
        return ds.map_batches(
            preprocess, 
            batch_format = "pandas")

In [ ]:
import torch
import torch.nn as nn
from transformers import BertModel
from transformers import AutoTokenizer

In [ ]:
model_name = "allenai/scibert_scivocab_uncased"
tokenizer = AutoTokenizer.from_pretrained(model_name)
llm = BertModel.from_pretrained(model_name, return_dict = False)
embedding_dim = llm.config.hidden_size
num_classes = 4

In [ ]:
class FinetunedLLM(nn.Module):
    def __init__(self, llm, dropout_p, embedding_dim, num_classes):
        super(FinetunedLLM, self).__init__()
        self.llm = llm
        self.dropout_p = dropout_p
        self.embedding_dim = embedding_dim
        self.num_classes = num_classes
        self.dropout = torch.nn.Dropout(dropout_p)
        self.fc1 =torch.nn.Linear(embedding_dim, num_classes)

    def forward(self, batch):
        ids, masks = batch["ids"], batch["masks"]
        seq, pool = self.llm(input_ids = ids, attention_mask = masks)
        z = self.dropout(pool)
        z = self.fc1(z)
        return z
    
    @torch.inference_mode()
    def predict(self,batch):
        self.eval()
        z = self(batch)
        y_pred = torch.argmax(z, dim=1).cpu().numpy()
        return y_pred
    
    def save(self, dp):
        with open(Path(dp, "args.json"), "w") as fp:
            contents = {
                "dropout_p": self.dropout_p,
                "embedding_dim": self.embedding_dim,
                "num_classes": self.num_classes,
            }
            json.dump(contents, fp, indent=4, sort_keys=False)
        torch.save(self.state_dict(), os.path.join(dp, "model.pt"))

    @classmethod
    def load(cls, args_fp, state_dict_fp):
        with open(args_fp, "r") as fp:
            kwargs = json.load(fp = fp)
        llm = BertModel.from_pretrained(model_name, return_dict = False)
        model = cls(llm = llm, **kwargs)
        model.load_state_dict(torch.load(state_dict_fp, map_location=torch.device("cpu")))
        return model 

In [ ]:
model = FinetunedLLM(llm=llm, dropout_p=0.5, embedding_dim=embedding_dim, num_classes=num_classes)
print(model.named_parameters)

In [ ]:
from ray.train.torch import get_device

In [ ]:
def pad_array(arr, dtype=np.int32):
    max_len = max(len(row) for row in arr)
    padded_arr = np.zeros((arr.shape[0], max_len), dtype=dtype)
    for i, row in enumerate(arr):
        padded_arr[i][:len(row)] = row
    return padded_arr

In [ ]:
def collate_fn(batch):
    batch["ids"] = pad_array(batch["ids"])
    batch["masks"] = pad_array(batch["masks"])
    dtypes = {"ids": torch.int32, "masks": torch.int32, "targets": torch.int64}
    tensor_batch = {}
    for key, array in batch.items():
        tensor_batch[key] = torch.as_tensor(array, dtype=dtypes[key], device=get_device())
    return tensor_batch

In [ ]:
from pathlib import Path 
import ray.train as train
from ray.train import Checkpoint, CheckpointConfig, DataConfig, RunConfig, ScalingConfig
from ray.train.torch import TorchCheckpoint, TorchTrainer, TorchConfig
import tempfile
import torch.nn.functional as F
from torch.nn.parallel.distributed import DistributedDataParallel

In [ ]:
def train_step(ds, batch_size, model, num_classes, loss_fn, optimizer):
    model.train()
    loss = 0.0
    ds_generator = ds.iter_torch_batches(batch_size=batch_size, collate_fn=collate_fn)
    for i, batch in enumerate(ds_generator):
        optimizer.zero_grad()
        z = model(batch)
        targets = F.one_hot(batch["targets"], num_classes=num_classes).float()
        J = loss_fn(z, targets)
        J.backward()
        optimizer.step()
        loss += (J.detach().item() - loss) / (i+1)
    return loss


In [ ]:
def eval_step(ds, batch_size, model, num_classes, loss_fn):
    model.eval()
    loss = 0.0 
    y_trues, y_preds = [], []
    ds_generator = ds.iter_torch_batches(batch_size=batch_size, collate_fn=collate_fn)
    with torch.inference_mode():
        for i, batch in enumerate(ds_generator):
            z = model(batch)
            targets = F.one_hot(batch["targets"], num_classes=num_classes).float()
            J = loss_fn(z, targets).item()
            loss += (J-loss)/(i+1)
            y_trues.extend(batch["targets"].cpu().numpy())
            y_preds.extend(torch.argmax(z, dim=1).cpu().numpy())
        return loss, np.vstack(y_trues), np.vstack(y_preds)

In [ ]:
def train_loop_per_worker(config):
    dropout_p = config["dropout_p"]
    lr = config["lr"]
    lr_factor = config["lr_factor"]
    lr_patience = config["lr_patience"]
    num_epochs = config["num_epochs"]
    batch_size = config["batch_size"]
    num_classes = config["num_classes"]

    setting_seeds()
    train_ds = train.get_dataset_shard("train")
    val_ds = train.get_dataset_shard("val")
    
    llm = BertModel.from_pretrained("allenai/scibert_scivocab_uncased", return_dict = False)
    model = FinetunedLLM(llm=llm, dropout_p=dropout_p, embedding_dim=llm.config.hidden_size, num_classes=num_classes)
    model = train.torch.prepare_model(model)

    loss_fn = nn.BCEWithLogitsLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode="min", factor=lr_factor, patience=lr_patience)

    num_workers = train.get_context().get_world_size()
    batch_size_per_worker = batch_size // num_workers
    for epoch in range(num_epochs):
        train_loss = train_step(train_ds, batch_size_per_worker, model, num_classes, loss_fn, optimizer)
        val_loss, _, _ = eval_step(val_ds, batch_size_per_worker, model, num_classes, loss_fn)
        scheduler.step(val_loss)

        with tempfile.TemporaryDirectory() as dp:
            if isinstance(model, DistributedDataParallel):
                model.module.save(dp=dp)
            else:
                model.save(dp=dp)
            metrics = dict(epoch=epoch, lr=optimizer.param_groups[0]["lr"], train_loss=train_loss, val_loss=val_loss)
            checkpoint = Checkpoint.from_directory(dp)
            train.report(metrics, checkpoint=checkpoint)

In [ ]:
from src.config import EFS_DIR, BASE_DIR, DATA_DIR, ARTIFACTS_DIR

In [ ]:
train_loop_config = {
    "dropout_p": 0.5,
    "lr": 1e-4,
    "lr_factor": 0.8,
    "lr_patience": 3,
    "num_epochs": 10,
    "batch_size": 256,
    "num_classes": num_classes,
}

In [ ]:
scaling_config = ScalingConfig(
    num_workers=num_workers,
    use_gpu=bool(resources_per_worker["GPU"]),
    resources_per_worker=resources_per_worker
)

In [ ]:
checkpoint_config = CheckpointConfig(num_to_keep=1, checkpoint_score_attribute="val_loss", checkpoint_score_order="min")
run_config = RunConfig(name="llm", checkpoint_config=checkpoint_config, storage_path=EFS_DIR)

In [ ]:
ds = data(dataset_loc=dataset)
train_ds, val_ds = stratify_split(ds, stratify="label", test_size=test_size)

preprocessor = CustomPreprocessor()
preprocessor = preprocessor.fitting(train_ds)
# materialize now, while the full cluster is free, so trainer.fit() doesn't have to
# tokenize on the fly while competing with the training worker for CPU
train_ds = preprocessor.transforming(train_ds).materialize()
val_ds = preprocessor.transforming(val_ds).materialize()

In [ ]:
trainer = TorchTrainer(
    train_loop_per_worker=train_loop_per_worker,
    train_loop_config=train_loop_config,
    scaling_config=scaling_config,
    run_config=run_config,
    datasets={"train": train_ds, "val": val_ds},
    dataset_config=DataConfig(datasets_to_split=["train"]),
    torch_config=TorchConfig(backend="gloo"),
)
results = trainer.fit()
results

In [ ]:
best_checkpoint = results.best_checkpoints[0][0]
best_metrics = results.best_checkpoints[0][1]
print(f"best checkpoint: {best_checkpoint.path}")
print(f"best metrics: {best_metrics}")

## Stop now, upload epoch-0 checkpoint (closing session before training finishes)

Use this instead of the `results`/`best_checkpoint` path — `results` only exists if `trainer.fit()` has actually returned, which won't be true if you stop training early.

**Before running the next cell:** interrupt/stop the running training cell in this notebook. Safe to do — the epoch-0 checkpoint below is already fully written to disk regardless of whether the trainer loop keeps going.

Requires `HF_TOKEN` (write access) and `HF_REPO_ID` (e.g. `yourname/agnews-scibert-classifier`) set in `.env`.

In [ ]:
import os
from huggingface_hub import create_repo, upload_folder

# points directly at the epoch-0 checkpoint already saved to disk
# (currently the best val_loss of any epoch trained so far)
checkpoint_dir = str(EFS_DIR / "llm" / "checkpoint_2026-08-10_13-03-12.020315")

HF_REPO_ID = os.environ["HF_REPO_ID"]  # e.g. "yourname/agnews-scibert-classifier", set in .env

create_repo(HF_REPO_ID, exist_ok=True, repo_type="model", private=True)
upload_folder(
    folder_path=checkpoint_dir,
    repo_id=HF_REPO_ID,
    repo_type="model",
    commit_message="Upload epoch 0 checkpoint (best val_loss so far)",
)
print(f"Uploaded to https://huggingface.co/{HF_REPO_ID}")

## Evaluate the uploaded checkpoint

Downloads the epoch-0 checkpoint back from Hugging Face and scores it on `val_ds`, so we get real metrics without waiting on a full retrain.

Runs standalone on CPU/whatever device is available — it does **not** reuse the training `collate_fn`, because that one calls `ray.train.torch.get_device()`, which only works inside an active Ray Train worker context. Everything else (`val_ds`, `num_classes`, `label_decoder`, `FinetunedLLM`, `pad_array`) is reused from the cells above, so run this after the data-prep and model-definition cells have executed at least once in this session.

Requires `HF_TOKEN` and `HF_REPO_ID` in `.env` (same as the upload step).

In [ ]:
import os
from huggingface_hub import snapshot_download

HF_REPO_ID = os.environ["HF_REPO_ID"]
eval_checkpoint_dir = snapshot_download(repo_id=HF_REPO_ID, repo_type="model")
print(f"Downloaded checkpoint to {eval_checkpoint_dir}")

In [ ]:
eval_model = FinetunedLLM.load(
    args_fp=Path(eval_checkpoint_dir, "args.json"),
    state_dict_fp=Path(eval_checkpoint_dir, "model.pt"),
)
eval_device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
eval_model = eval_model.to(eval_device)
print(f"Loaded checkpoint onto {eval_device}")

In [ ]:
def eval_collate_fn(batch):
    batch["ids"] = pad_array(batch["ids"])
    batch["masks"] = pad_array(batch["masks"])
    dtypes = {"ids": torch.int32, "masks": torch.int32, "targets": torch.int64}
    return {key: torch.as_tensor(array, dtype=dtypes[key], device=eval_device) for key, array in batch.items()}

def evaluate_checkpoint(ds, batch_size, model, num_classes):
    model.eval()
    loss_fn = nn.BCEWithLogitsLoss()
    loss = 0.0
    y_trues, y_preds = [], []
    ds_generator = ds.iter_torch_batches(batch_size=batch_size, collate_fn=eval_collate_fn)
    model_device = next(model.parameters()).device
    with torch.inference_mode():
        for i, batch in enumerate(ds_generator):
            batch = {key: value.to(model_device) for key, value in batch.items()}
            z = model(batch)
            targets = F.one_hot(batch["targets"], num_classes=num_classes).float()
            J = loss_fn(z, targets).item()
            loss += (J - loss) / (i + 1)
            y_trues.extend(batch["targets"].cpu().numpy())
            y_preds.extend(torch.argmax(z, dim=1).cpu().numpy())
    return loss, np.array(y_trues), np.array(y_preds)

In [ ]:
from sklearn.metrics import classification_report

val_loss, y_true, y_pred = evaluate_checkpoint(val_ds, batch_size=64, model=eval_model, num_classes=num_classes)
print(f"val_loss: {val_loss:.4f}")
print(classification_report(y_true, y_pred, target_names=[label_decoder[i] for i in sorted(label_decoder)]))